# UNSW-NB15 Linear Transformer ONNX Export

This notebook loads the completed Linear Transformer baseline from `best_model.pt`,
exports a fixed-shape ONNX model, and verifies PyTorch/ONNXRuntime logits.

It does not train, optimize, synthesize, or run on hardware.

## 2. Import dependencies

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("ONNX:", onnx.__version__)
print("ONNXRuntime:", ort.__version__)

Python: 3.10.12
PyTorch: 1.13.1+cu116
ONNX: 1.13.0
ONNXRuntime: 1.16.1


## 3. Path configuration

In [2]:
PROJECT_ROOT = Path("/home/cym/prj2/finn/notebooks/icl_thesis-master")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.export_unsw_linear_onnx import (
    CHECKPOINT_PATH,
    MODEL_CONFIG_PATH,
    ONNX_DIR,
    ONNX_MODEL_PATH,
    SAMPLE_INPUTS_PATH,
    SAMPLE_LABELS_PATH,
    LinearUNSWAnomalyDetector,
    check_onnx_model,
    create_onnx_session,
    error_metrics,
    export_onnx,
    extract_state_dict,
    load_json,
    load_preprocess_info,
    load_samples,
    onnx_inference_fixed_batch,
    pytorch_inference,
    save_validation_results,
    torch_load_compatible,
)

ONNX_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint:", CHECKPOINT_PATH)
print("Sample inputs:", SAMPLE_INPUTS_PATH)
print("Sample labels:", SAMPLE_LABELS_PATH)
print("ONNX output:", ONNX_MODEL_PATH)

Matplotlib created a temporary config/cache directory at /tmp/matplotlib-05j9429h because the default path (/home/cym/.config/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Checkpoint: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/best_model.pt
Sample inputs: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/sample_inputs.npy
Sample labels: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/sample_labels.npy
ONNX output: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/onnx/unsw_linear_transformer.onnx


## 4. Read preprocessing metadata

In [3]:
preprocess_info = load_preprocess_info()
expected_sample_shape = (
    int(preprocess_info["seq_len"]),
    int(preprocess_info["input_dim"]),
)

print("Preprocessed feature dimension:", preprocess_info["one_hot_feature_dim"])
print("Expected sample shape:", expected_sample_shape)
print("Expected model input shape:", preprocess_info["model_input_shape"])

Preprocessed feature dimension: 192
Expected sample shape: (8, 24)
Expected model input shape: [None, 8, 24]


## 5. Read saved inputs and labels

In [4]:
sample_inputs, sample_labels = load_samples(expected_sample_shape)
one_input = sample_inputs[:1]
first100_inputs = sample_inputs[: min(100, len(sample_inputs))]

print("sample_inputs shape:", sample_inputs.shape)
print("sample_inputs dtype:", sample_inputs.dtype)
print("sample_labels shape:", sample_labels.shape)
print("single export input shape:", one_input.shape)
print("validation input count:", len(first100_inputs))

sample_inputs shape: (256, 8, 24)
sample_inputs dtype: float32
sample_labels shape: (256,)
single export input shape: (1, 8, 24)
validation input count: 100


## 6. Rebuild the training-time Linear Transformer

In [5]:
checkpoint = torch_load_compatible(CHECKPOINT_PATH)
state_dict, checkpoint_model_config, checkpoint_format = extract_state_dict(checkpoint)
model_config = checkpoint_model_config or load_json(MODEL_CONFIG_PATH)
model = LinearUNSWAnomalyDetector(**model_config).cpu()

print("Model class:", model.__class__.__name__)
print("Checkpoint format:", checkpoint_format)
print("Model config:", json.dumps(model_config, indent=2))
print(model)

Model class: LinearUNSWAnomalyDetector
Checkpoint format: model_state_dict
Model config: {
  "input_dim": 24,
  "seq_len": 8,
  "d_model": 16,
  "dim_feedforward": 32,
  "num_layers": 1,
  "dropout": 0.1,
  "num_classes": 2
}
LinearUNSWAnomalyDetector(
  (input_projection): Linear(in_features=24, out_features=16, bias=True)
  (layers): ModuleList(
    (0): _UNSWLinearEncoderBlock(
      (attention): _UNSWLinearAttention(
        (query): Linear(in_features=16, out_features=16, bias=False)
        (key): Linear(in_features=16, out_features=16, bias=False)
        (value): Linear(in_features=16, out_features=16, bias=False)
        (output): Linear(in_features=16, out_features=16, bias=True)
      )
      (feedforward): Sequential(
        (0): Linear(in_features=16, out_features=32, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=32, out_features=16, bias=True)
      )
      (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine

## 7. Load `best_model.pt`

In [6]:
load_result = model.load_state_dict(state_dict, strict=True)
model.eval()

print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)
print("Model device:", next(model.parameters()).device)
print("Training mode:", model.training)

Missing keys: []
Unexpected keys: []
Model device: cpu
Training mode: False


## 8. PyTorch single-sample inference

In [7]:
pytorch_one_logits = pytorch_inference(model, one_input)
pytorch_first100_logits = pytorch_inference(model, first100_inputs)
pytorch_prediction = int(np.argmax(pytorch_one_logits, axis=1)[0])

print("Input shape:", one_input.shape)
print("Logits shape:", pytorch_one_logits.shape)
print("PyTorch logits:", pytorch_one_logits)
print("Predicted label:", pytorch_prediction)
print("True label:", int(sample_labels[0]))

Input shape: (1, 8, 24)
Logits shape: (1, 2)
PyTorch logits: [[-1.3556118  1.2704847]]
Predicted label: 1
True label: 0


## 9. Export fixed-shape ONNX

In [8]:
opset_version = export_onnx(model, one_input)
print("Exported opset version:", opset_version)
print("ONNX model path:", ONNX_MODEL_PATH)

Dummy input shape: (1, 8, 24)
ONNX saved to: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/onnx/unsw_linear_transformer.onnx
Exported opset version: 13
ONNX model path: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/onnx/unsw_linear_transformer.onnx


/home/cym/prj2/finn/notebooks/icl_thesis-master/src/transformer.py:665: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.size(1) != self.seq_len or x.size(2) != self.input_dim:
/usr/local/lib/python3.10/dist-packages/torch/onnx/_internal/jit_utils.py:258: UserWarning: The shape inference of prim::Constant type is missing, so it may result in wrong shape inference for the exported graph. Please consider adding it in symbolic function. (Triggered internally at ../torch/csrc/jit/passes/onnx/shape_type_inference.cpp:1884.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/usr/local/lib/python3.10/dist-packages/torch/onnx/utils.py:687: UserWarning: The shape inference of prim::Constant type is missing, so it may result in wrong shape inferen

## 10. Run `onnx.checker` and collect operator statistics

In [9]:
onnx_model, operator_counts = check_onnx_model()
actual_opset_version = int(onnx_model.opset_import[0].version)
if actual_opset_version != opset_version:
    raise RuntimeError(
        f"Expected ONNX opset {opset_version}, found {actual_opset_version}"
    )

operator_stats = pd.DataFrame(
    sorted(operator_counts.items()), columns=["op_type", "count"]
)
print("onnx.checker: passed")
print("ONNX input:", onnx_model.graph.input[0].name)
print("ONNX output:", onnx_model.graph.output[0].name)
display(operator_stats)

onnx.checker: passed
ONNX input: input
ONNX output: logits


,op_type,count
0,Add,15
1,Clip,1
2,Constant,11
3,Div,4
4,Elu,2
5,Gemm,1
6,MatMul,10
7,Mul,3
8,Pow,3
9,ReduceMean,7


## 11. ONNXRuntime inference

In [10]:
session = create_onnx_session()
onnx_one_logits = onnx_inference_fixed_batch(session, one_input)
onnx_first100_logits = onnx_inference_fixed_batch(session, first100_inputs)

print("Session providers:", session.get_providers())
print("ONNX input shape:", session.get_inputs()[0].shape)
print("ONNX output shape:", session.get_outputs()[0].shape)
print("ONNX single-sample logits:", onnx_one_logits)

Session providers: ['CPUExecutionProvider']
ONNX input shape: [1, 8, 24]
ONNX output shape: [1, 2]
ONNX single-sample logits: [[-1.355612   1.2704852]]


## 12. Verify PyTorch and ONNX logits

In [11]:
single_errors = error_metrics(pytorch_one_logits, onnx_one_logits)
first100_errors = error_metrics(pytorch_first100_logits, onnx_first100_logits)
onnx_prediction = int(np.argmax(onnx_one_logits, axis=1)[0])
pytorch_first100_predictions = np.argmax(pytorch_first100_logits, axis=1)
onnx_first100_predictions = np.argmax(onnx_first100_logits, axis=1)
prediction_match_rate = float(
    np.mean(pytorch_first100_predictions == onnx_first100_predictions)
)
logits_allclose = bool(
    np.allclose(
        pytorch_first100_logits,
        onnx_first100_logits,
        rtol=1e-4,
        atol=1e-5,
    )
)

print("PyTorch one-sample logits:", pytorch_one_logits)
print("ONNX one-sample logits:", onnx_one_logits)
print("Single-sample errors:", single_errors)
print("PyTorch predicted label:", pytorch_prediction)
print("ONNX predicted label:", onnx_prediction)
print("True label:", int(sample_labels[0]))
print("Single prediction match:", pytorch_prediction == onnx_prediction)
print("First-100 errors:", first100_errors)
print("First-100 prediction match rate:", prediction_match_rate)
print("First-100 logits allclose:", logits_allclose)

if not logits_allclose or prediction_match_rate != 1.0:
    raise AssertionError("PyTorch and ONNX outputs did not pass consistency validation")

PyTorch one-sample logits: [[-1.3556118  1.2704847]]
ONNX one-sample logits: [[-1.355612   1.2704852]]
Single-sample errors: {'max_abs_error': 4.76837158203125e-07, 'mean_abs_error': 3.5762786865234375e-07, 'rmse': 3.769728732309794e-07}
PyTorch predicted label: 1
ONNX predicted label: 1
True label: 0
Single prediction match: True
First-100 errors: {'max_abs_error': 1.0728836059570312e-06, 'mean_abs_error': 2.9675662517547606e-07, 'rmse': 3.9339206660985735e-07}
First-100 prediction match rate: 1.0
First-100 logits allclose: True


## 13. Save ONNX outputs and validation reports

In [12]:
validation_report = save_validation_results(
    model_config=model_config,
    checkpoint_format=checkpoint_format,
    opset_version=opset_version,
    op_counts=operator_counts,
    labels=sample_labels,
    pytorch_one=pytorch_one_logits,
    onnx_one=onnx_one_logits,
    pytorch_first100=pytorch_first100_logits,
    onnx_first100=onnx_first100_logits,
)

required_outputs = [
    ONNX_MODEL_PATH,
    ONNX_DIR / "onnx_graph_stats.csv",
    ONNX_DIR / "onnx_validation_report.json",
    ONNX_DIR / "onnx_validation_summary.md",
    ONNX_DIR / "pytorch_one_sample_logits.npy",
    ONNX_DIR / "onnx_one_sample_logits.npy",
    ONNX_DIR / "pytorch_first100_logits.npy",
    ONNX_DIR / "onnx_first100_logits.npy",
]
for path in required_outputs:
    if not path.is_file():
        raise FileNotFoundError(f"Expected output was not created: {path}")

print("Can proceed to HLS/Vivado/PYNQ preparation:", validation_report["can_proceed_to_hls_vivado_pynq"])
print("Saved files:")
for path in required_outputs:
    print(f" - {path.name}: {path.stat().st_size} bytes")

Can proceed to HLS/Vivado/PYNQ preparation: True
Saved files:
 - unsw_linear_transformer.onnx: 20037 bytes
 - onnx_graph_stats.csv: 150 bytes
 - onnx_validation_report.json: 1754 bytes
 - onnx_validation_summary.md: 1481 bytes
 - pytorch_one_sample_logits.npy: 136 bytes
 - onnx_one_sample_logits.npy: 136 bytes
 - pytorch_first100_logits.npy: 928 bytes
 - onnx_first100_logits.npy: 928 bytes
